In [ ]:
# 실습 준비 — 12주차 통계적 추정 · 통계적 가설검정
# 이 셀을 먼저 한 번 실행하세요. 데이터가 없으면 아래 셀들이 전부 실패합니다.
import os, pathlib, urllib.request

BASE = "https://raw.githubusercontent.com/aprilslab/statistics-lab/main/data/"
FILES = ["ch11_potato.csv", "ch11_training_rel.csv", "ch11_training_ind.csv", "ch11_ad.csv"]

pathlib.Path("data").mkdir(exist_ok=True)
for name in FILES:
    for dest in (pathlib.Path(name), pathlib.Path("data") / name):
        if not dest.exists():
            urllib.request.urlretrieve(BASE + name, dest)

# '../data/x.csv' 로 읽는 노트북 대응 — 상위 폴더에도 같은 data/ 를 걸어둔다.
# 절대경로(/data)로 박으면 cwd 가 /content 가 아닐 때 깨지므로 상대경로로 건다.
try:
    parent = pathlib.Path("..") / "data"
    if not parent.exists():
        os.symlink(pathlib.Path("data").resolve(), parent)
except OSError:
    pass

print("준비 완료:", ", ".join(FILES) if FILES else "(내려받을 데이터 없음)")


# 12주차 · 통계적 가설검정 — 실습

수업 슬라이드의 코드를 슬라이드 순서대로 따라 합니다. 맨 위 준비 셀부터 차례로 실행하세요.

`{ }` 가 있는 셀은 빈칸을 채운 뒤 실행합니다. 셀 첫 주석에 무엇을 채울지 적어 두었고, 줄 끝 주석의 값이 슬라이드와 같은 결과입니다.

In [ ]:
# 준비 — 실행만 하세요
import numpy as np
import pandas as pd
from scipy import stats
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'   # 셀 안의 결과를 줄마다 모두 보인다
np.set_printoptions(legacy='1.25')                 # 70.4 처럼 숫자만 보인다

## p값 — 코드로 구해 유의수준과 비교한다

검정통계량보다 극단적인 쪽 영역의 면적을 p값이라 한다(교재 299쪽) — 귀무가설이 옳다고 할 때 관측한 값 또는 그보다 더 극단적인 값이 나올 확률이다. p값이 유의수준보다 작으면 귀무가설을 기각하고, 그렇지 않으면 채택한다. 모분산을 알 때 모평균의 검정통계량은 Z 다.

In [ ]:
# 슬라이드 8장 — p값 — 코드로 구해 유의수준과 비교한다
# 빈칸 { } — p값 — 표준정규분포에서 z 보다 왼쪽의 넓이를 주는 함수
sample = np.array(pd.read_csv('ch11_potato.csv')['무게'])   # 감자튀김
n = len(sample)                                  # 14
z = (np.mean(sample) - 130) / np.sqrt(9 / n)     # −1.932   검정통계량 (μ₀ = 130, σ² = 9)
stats.norm().{ }(z)                              # 0.027    p값 — 0.027 < 0.05 → 귀무가설 기각

## 모평균의 검정 — 모분산을 모를 때 (1표본 t 검정)

언제: 모평균이 어떤 값 μ₀ 인지 검정하는데 모분산을 모른다(실제 분석의 대부분). 모집단이 정규분포라는 가정이 필요하다. σ² 대신 불편분산 s² 로 표준화한 t 는 귀무가설 아래에서 자유도 n − 1 인 t 분포를 따른다(교재 311쪽).

In [ ]:
# 슬라이드 15장 — 모평균의 검정 — 모분산을 모를 때 (1표본 t 검정)
# 빈칸 { } — 모분산을 모를 때 1표본 t 검정 함수
stats.{ }(sample, 130)    # t = −1.455, p = 0.169   (양측) — H₀: μ = 130

## 모분산의 검정 — 카이제곱분포

언제: 모분산이 어떤 값 σ₀² 인지 검정한다. 모집단이 정규분포라는 가정이 필요하다. 불편분산으로 만든 Y 는 귀무가설 아래에서 자유도 n − 1 인 카이제곱분포를 따른다 — 모분산의 구간추정에서 쓴 바로 그 Y 다(교재 310쪽).

In [ ]:
# 슬라이드 16장 — 모분산의 검정 — 카이제곱분포
# 빈칸 { } — 카이제곱분포의 자유도 — n − 1
n = len(sample)                         # 14
u_var = np.var(sample, ddof=1)          # 15.869   불편분산 s²
y = (n - 1) * u_var / 9                 # 22.922   검정통계량 (σ₀² = 9)
rv = stats.chi2(df={ })
2 * min(rv.cdf(y), rv.sf(y))            # 0.085    p값 (양측 — y 쪽 꼬리 넓이 × 2)

## 대응비교 t 검정 — stats.ttest_rel

대응비교 t 검정이란 대응하는 데이터가 있고, 데이터 차이에 정규분포를 가정할 수 있는 경우의 평균값 차이에 대한 검정이다(교재 315쪽). 차이 D = 후 − 전 을 표본 하나로 보면, 차이의 평균이 0 인지 보는 1표본 t 검정과 같다.

In [ ]:
# 슬라이드 18장 — 대응비교 t 검정 — stats.ttest_rel
# 빈칸 { } — 대응비교 t 검정 함수
rel = pd.read_csv('ch11_training_rel.csv')       # 전 · 후 20쌍
stats.{ }(rel['후'], rel['전'])              # t = 2.204, p = 0.040
stats.ttest_1samp(rel['후'] - rel['전'], 0)         # 같은 값 — 차이의 1표본 t 검정

## 독립비교 t 검정 — stats.ttest_ind(equal_var=False)

독립비교 t 검정이란 대응하는 데이터가 없고 독립된 2표본 모집단에 정규분포를 가정할 수 있는 경우 평균값의 차이에 대한 검정이다(교재 318쪽). 두 모분산이 같다고 가정하지 않는 방법을 웰치의 방법이라 한다(320쪽) — 코드의 equal_var=False.

In [ ]:
# 슬라이드 19장 — 독립비교 t 검정 — stats.ttest_ind(equal_var=False)
# 빈칸 { } — 두 학급의 분산이 같다고 가정하지 않는 인자 (웰치)
ind = pd.read_csv('ch11_training_ind.csv')                  # A · B 학급 각 20명
stats.ttest_ind(ind['A'], ind['B'], equal_var={ })       # t = −1.761, p = 0.087   (웰치)

## 윌콕슨의 부호순위검정 — stats.wilcoxon

윌콕슨의 부호순위검정은 대응표본에서 차이에 정규분포를 가정할 수 없는 경우, 중앙값의 차이에 대한 검정이다(교재 320쪽). 이렇게 읽으려면 차이의 모집단 분포가 좌우 대칭이라는 가정이 필요하다(귀무가설은 그 중심이 0 이라는 것) — 정규분포일 필요는 없다. 차이의 값 대신 순위를 쓴다(아래는 근력운동 첫 6쌍).

In [ ]:
# 슬라이드 22장 — 윌콕슨의 부호순위검정 — stats.wilcoxon
# 빈칸 { } — 윌콕슨의 부호순위검정 함수
stats.{ }(rel['후'] - rel['전'])    # 통계량 49.5, p = 0.038 (양측) → 기각 — 대응비교 t 검정(p 0.040)과 같은 결론

## 만·위트니의 U 검정 — stats.mannwhitneyu

만·위트니의 U 검정은 대응되는 데이터가 없는 2표본 모집단에 정규분포를 가정할 수 없는 경우, 중앙값의 차이에 대한 검정이다. 윌콕슨의 순위합검정이라고도 부른다(교재 327쪽). 일반적으로는 한쪽이 더 큰 값을 내는 경향이 있는지 두 분포를 비교하는 검정이고, 중앙값의 차이로 읽으려면 두 분포의 모양 · 퍼짐이 같고 위치만 다르다는 조건이 붙는다.

In [ ]:
# 슬라이드 23장 — 만·위트니의 U 검정 — stats.mannwhitneyu
# 빈칸 { } — 만·위트니의 U 검정 함수
stats.{ }(ind['A'], ind['B'], alternative='two-sided')   # U = 130, p = 0.059 → 채택 — 독립비교 t 검정(p 0.087)과 같은 결론

## 카이제곱검정(독립성 검정) — stats.chi2_contingency

독립성 검정이란 두 변수 X 와 Y 에 관해서 'X 와 Y 가 독립이다' 라는 귀무가설과 'X 와 Y 가 독립이 아니다' 라는 대립가설에 의해 수행되는 검정이다(교재 331쪽). 독립일 때 기대되는 도수를 기대도수, 실제로 관측된 도수를 관측도수라 한다(334쪽). 조건: 관측(사람)끼리 독립이고, 기대도수가 충분히 클 것(보통 모든 칸 5 이상) — 카이제곱분포는 근사다.

In [ ]:
# 슬라이드 26장 — 카이제곱검정(독립성 검정) — stats.chi2_contingency
# 빈칸 { } — 독립성 검정 함수 — 교차집계표를 넣는다
ad = pd.read_csv('ch11_ad.csv')                       # 1000명: 본 광고 · 구입 여부
cross = pd.crosstab(ad['광고'], ad['구입'])            # 교차집계표
stats.{ }(cross, correction=False)       # 통계량 3.750, p = 0.053, 자유도 1 → 채택